# OpenAI Responses provider

Goal: construct `Agent.openai` for the official OpenAI Responses API, then optionally run a live model and tool call with an explicit `api_key`.

Trust: T1 native provider plus T2 Python ports. Network: skipped unless the next cell or `OPENAI_API_KEY` has a key.

`api_key` is required and keyword-only. The binding does not read environment variables. Paste a key below, or keep using the environment.

Set `OPENAI_MODEL` to an official OpenAI model id. `reasoning_effort` is optional (`none`, `minimal`, `low`, `medium`, `high`, `xhigh`, `max`), and `reasoning_summary` is optional (`auto`, `concise`, `detailed`). Requests are stateless (`store: false`) and tool continuations replay typed Responses output items.

In [ ]:
OPENAI_API_KEY = ""  # paste a valid key to run the live cells; never commit it
OPENAI_MODEL = "gpt-5.6-luna"
OPENAI_REASONING_EFFORT = "low"  # or none / minimal / medium / high / xhigh / max
OPENAI_REASONING_SUMMARY = "auto"  # or concise / detailed

In [7]:
from _support import live_value

import finstack_ai

api_key = live_value(OPENAI_API_KEY, "OPENAI_API_KEY")
if api_key:
    agent = await finstack_ai.Agent.openai(
        OPENAI_MODEL,
        "Answer concisely.",
        api_key=api_key,
        reasoning_effort=OPENAI_REASONING_EFFORT or None,
        reasoning_summary=OPENAI_REASONING_SUMMARY or None,
    )
    print(agent.compact_capability_catalog())
else:
    print("skipped: OPENAI_API_KEY unset")

`Agent.openai` targets only the official OpenAI Responses endpoint and requires `api_key=`. Local models use `Agent.ollama` and Ollama's native `/api/chat` endpoint, as shown in notebook 05.

The live cell uses `OPENAI_MODEL`, `OPENAI_REASONING_EFFORT`, and `OPENAI_REASONING_SUMMARY` from the first code cell. Responses supports reasoning and function tools in the same agent.

In [6]:
from pydantic import BaseModel


class Answer(BaseModel):
    answer: int


@finstack_ai.tool
def add(left: int, right: int) -> Answer:
    """Add two integers."""
    return Answer(answer=left + right)


tools = finstack_ai.pydantic_toolset(
    add,
    component="notebook.toolset.openai",
    name="math",
)

# A rejected or revoked key causes the run to fail before model output begins.
if api_key:
    reasoning_result = await agent.run(
        "What is 20 plus 22? Reply with only the number."
    )
    print("reasoning:", reasoning_result.text)

    tool_agent = await finstack_ai.Agent.openai(
        OPENAI_MODEL,
        "Answer with the tool result only.",
        api_key=api_key,
        reasoning_effort=OPENAI_REASONING_EFFORT or None,
        reasoning_summary=OPENAI_REASONING_SUMMARY or None,
        toolsets=[tools],
    )
    tool_result = await tool_agent.run("Add 20 and 22")
    print("tool:", tool_result.text)
else:
    print("skipped: OPENAI_API_KEY unset")

reasoning: 42
tool: 42
